### ABOUT ME
This script builds up the fine tuning prompts used by the Ministral Mental and Ministral Mappy Models.

It essentially tries to tie each training set item to specific defenses and items.

In an attempt to deal with the class imbalance, each non 0,7 label is ran through the script again.

Don't run me, this script takes several hours to finish!

In [1]:
import json
import re

from mappyutils.dmrsq_util import *

In [2]:
dmrsq_file_path = r"./dmrsq_items_json.json"
training_dialogue_with_items_path = r"./training_with_items_added_updated.json"

with open(dmrsq_file_path, "r", encoding="utf-8") as f:
    dmrsqData = json.load(f)

with open(training_dialogue_with_items_path, "r", encoding="utf-8") as f:
    trainingdata = json.load(f)


item_to_defense = {}
for defense, items in defense_map.items():
    for item_num in items:
        item_to_defense[f"ITEM_{item_num:03d}"] = defense

defense_to_level = {}
for level, dcodes in LEVEL_INDEX_MAP.items():
    # map defense codes back to defense names
    dcode_to_defense = {
        v: k
        for k, vals in zip(defense_map.keys(), [f"D{i}" for i in range(1, 31)])
        for v in vals
    }
    for defense, items in defense_map.items():
        defense_to_level[defense] = level


# DATA LOADING
def load_json(path):
    with open(path, "r", encoding="utf8") as f:
        return json.load(f)

In [3]:
from collections import defaultdict


def format_dmrs_answer(positive_nums, positive_defense_names):
    # Sort items by confidence score descending
    sorted_items = sorted(positive_nums.items(), key=lambda x: x[1], reverse=True)
    items_list_str = ", ".join(
        [dmrsqData[f"ITEM_{num:03d}"]["text"] for num, score in sorted_items]
    )

    # Format defense names in natural English
    defenses = sorted(positive_defense_names)
    num_defenses = len(defenses)
    defenses = [f'"{d}"' for d in defenses]

    if num_defenses == 0:
        defenses_str = ""
        defense_word = "defense"
        verb = "applies"
    elif num_defenses == 1:
        defenses_str = f"{defenses[0]}"
        defense_word = "defense"
        verb = "applies"
    elif num_defenses == 2:
        defenses_str = f"{defenses[0]} and {defenses[1]}"
        defense_word = "defenses"
        verb = "apply"
    else:
        defenses_str = ", ".join(defenses[:-1]) + ", and " + defenses[-1] + ""
        defense_word = "defenses"
        verb = "apply"

    # Final natural English answer
    formatted_answer = (
        f"DMRS {defense_word} {defenses_str} {verb}. "
        f"Reasoning: The final chat log and its prior context indicate these items: {items_list_str}."
    )

    return formatted_answer


def build_training_rows(trainingData):
    '''build up a set of training rows.'''
    rows = []
    no_7_labels = []
    found_defences = set()
    all_list_values = defaultdict(list)
    all_possible_items = set(dmrsqData.keys())
    for record in trainingData:
        ctx = []
        for turn in record["dialogue"]:
            if turn["speaker"] == "seeker":
                ctx.append(f"> seeker: {turn["text"]}")
            else:
                ctx.append(f"> supporter: {turn["text"]}")

        fullt = "\n".join(ctx)

        text = record["current_text"]
        label = record["label"]
        if label in [0]:
            prompt = f"Consider this chat log: ```{fullt}\n``` Question:What DMRS defenses apply to the seeker, prioritizing the log at the end?"
            answer = f"No DMRS defense applies, this is a purely functional utterance that serves a purely conversational or social function and does not engage with emotional conflict or psychological content. These are statements that are functionally neutral.  In Short, it's Level-0 \"NO DMRS DEFENSES.\""

        elif label == 8:
            outputs = ""

            prompt = f"Consider this chat log: ```{fullt}\n``` Question:What DMRS defenses apply to the seeker, prioritizing the log at the end?"
            answer = f"There is very weak evidence for this post.  Reasoning: The given posts are too unclear to assign any given defenses."

        else:
            outputs = ""
            positive_nums = {}
            for item_id, score in record["items_soft"].items():
                match = re.search(r"ITEM\s+(\d+)", item_id)
                if match:
                    item_num = int(match.group(1))
                else:
                    raise Exception("NO ITEM ID FOUND!")
                positive_nums[item_num] = score

            positive_defense_names = {
                dmrsqData[f"ITEM_{n:03d}"]["defense"]
                for n in positive_nums
                if f"ITEM_{n:03d}" in dmrsqData
            }
            for n in positive_nums:
                if f"ITEM_{n:03d}" in dmrsqData:
                    found_defences.add(dmrsqData[f"ITEM_{n:03d}"]["defense"])
                    if (
                        f"ITEM_{n:03d}"
                        not in all_list_values[dmrsqData[f"ITEM_{n:03d}"]["defense"]]
                    ):
                        all_list_values[dmrsqData[f"ITEM_{n:03d}"]["defense"]].append(
                            f"ITEM_{n:03d}"
                        )


            prompt = f"Consider this chat log: ```{fullt}\n``` Question:What DMRS defenses apply to the seeker, prioritizing the log at the end?"
            answer = format_dmrs_answer(positive_nums, positive_defense_names)
        rows.append((prompt, answer))
        no_7_labels.append((prompt, answer,label))

    # Flatten all items that were seen
    seen_items = set()
    for items in all_list_values.values():
        seen_items.update(items)

    # Compute missing items
    missing_items = all_possible_items - seen_items

    # Optional: group missing items by defense
    missing_by_defense = defaultdict(list)
    for item in missing_items:
        defense = dmrsqData[item]["defense"]
        missing_by_defense[defense].append(item)
    print(missing_by_defense)
    for defen, items in missing_by_defense.items():
        print(defen, items)
    return rows, no_7_labels

In [4]:
outs = build_training_rows(trainingdata)

defaultdict(<class 'list'>, {'Anticipation': ['ITEM_046'], 'Undoing': ['ITEM_083'], 'Suppression': ['ITEM_131'], 'Autistic fantasy': ['ITEM_002', 'ITEM_110'], 'Dissociation': ['ITEM_027', 'ITEM_073'], 'Passive aggression': ['ITEM_089'], 'Sublimation': ['ITEM_014', 'ITEM_036', 'ITEM_063'], 'Altruism': ['ITEM_079'], 'Omnipotence': ['ITEM_068'], 'Acting out': ['ITEM_005'], 'Repression': ['ITEM_136'], 'Humor': ['ITEM_051', 'ITEM_119']})
Anticipation ['ITEM_046']
Undoing ['ITEM_083']
Suppression ['ITEM_131']
Autistic fantasy ['ITEM_002', 'ITEM_110']
Dissociation ['ITEM_027', 'ITEM_073']
Passive aggression ['ITEM_089']
Sublimation ['ITEM_014', 'ITEM_036', 'ITEM_063']
Altruism ['ITEM_079']
Omnipotence ['ITEM_068']
Acting out ['ITEM_005']
Repression ['ITEM_136']
Humor ['ITEM_051', 'ITEM_119']


In [5]:
def compare_dicts(d1, d2):
    """
    Returns True if all checks pass, otherwise False.

    Rule update:
    If test #3 (nothing) and test #4 (defenses) both match AND both
    values for 'nothing' are not None, then item mismatch is ignored.

    Requires dmrsqData globally available.
    """

    items1 = set(d1.get("items", []))
    items2 = set(d2.get("items", []))

    r1 = d1.get("rating")
    r2 = d2.get("rating")

    n1 = d1.get("nothing")
    n2 = d2.get("nothing")

    rating_match=False
    # 2. Compare rating
    if r1 is not None or r2 is not None:
        if r1 != r2:
            print("FAILED: Rating mismatch")
            print("d1 rating:", r1)
            print("d2 rating:", r2)
            return False
        else:
            rating_match =True

    # 3. Compare nothing
    nothing_matches = True

    if n1 is not None or n2 is not None:
        if n1 != n2:
            print("FAILED: Nothing mismatch")
            print("d1 nothing:", n1)
            print("d2 nothing:", n2)
            return False

    # both explicitly non-null and equal
    both_nothing_present = (n1 is not None and n2 is not None)

    # 4. Validate defenses from items in d1
    matched_items = items1

    positive_defense_names = {
        dmrsqData[f"ITEM_{n:03d}"]["defense"]
        for n in matched_items
        if f"ITEM_{n:03d}" in dmrsqData
    }

    d2_defenses = {
        defense_name
        for _, defense_name in d2.get("defenses", [])
    }

    invalid_defenses = d2_defenses - positive_defense_names
    defenses_match = len(invalid_defenses) == 0

    if not defenses_match:
        print("FAILED: Invalid defenses found in d2")
        print("Allowed defenses:", sorted(positive_defense_names))
        print("d2 defenses:", sorted(d2_defenses))
        print("Invalid defenses:", sorted(invalid_defenses))
        print("Matched items:", sorted(matched_items))
        return False

    if items1 != items2:

        # ignore mismatch if test3 + test4 passed and nothing is non-null
        if rating_match and defenses_match:
            print("NOTICE: Item mismatch ignored because this is label 8")
            print("d1 items:", sorted(items1))
            print("d2 items:", sorted(items2))
            return True

        print("FAILED: Items do not match")
        print("d1 items only:", sorted(items1 - items2))
        print("d2 items only:", sorted(items2 - items1))
        print("d1 items:", sorted(items1))
        print("d2 items:", sorted(items2))
        return False

    return True

In [7]:
import pandas as pd
import ollama
from mappyutils.dr_mappy_output_eval import (
    parse_paragraph,
    post_process,
    print_post_process,
    save_json as save_json_external,
)

import random

# Roll 1d20
roll = random.randint(1, 20)
if roll < 3:
    N = 1
def save_rows_to_csv_pandas(rows, filename="dmrs_defenses_datasetb.csv"):
    df = pd.DataFrame(rows, columns=["query", "output"])
    df.to_csv(filename, index=False, encoding="utf-8")

# Build data
trainings, outs = build_training_rows(trainingdata)
trainings = []

for query, model_output, label in outs:
    if label == 0:
        trainings.append((query, model_output))


save_rows_to_csv_pandas(trainings, filename="defenses_level_zero.csv")


import random
from concurrent.futures import ThreadPoolExecutor, as_completed

trainings = []

label_to_name = {
    0: "No Defense",
    1: "Action Defense Level",
    2: "Major Image-distorting Defense Level",
    3: "Disavowal Defense Level",
    4: "Minor Image-distorting Defense Level",
    5: "Neurotic Defense Level",
    6: "Obsessional Defense Level",
    7: "Highly Adaptive Defense Level",
    8: "Needs More Evidence"
}
lima=932 
for i, tups in enumerate(outs[lima:]):
    query, expected_response,label=tups
    N=4

    # Multipliers are tuned so there's at least 300ish entries per label
    multipliers = {
        0: 1, 1: 3, 2: 6,  3: 4,   4: 4,
        5: 7, 6: 2,  7: 1,  8: 8
    }

    N = multipliers.get(label, 1)
    b=0
    tocomparewith=expected_response
    compare_dict=parse_paragraph(tocomparewith)
    
    post2 = post_process([compare_dict])
    print(post2)
    print(i+lima,label,compare_dict)
    
    redocount=0
    while b<N:
        #model_name = "hf.co/CrosswaveOmega/ministral8b-drmappy-mental-lora-unquantized-Q8_0-GGUF:Q8_0"
        model_name = "ministral-3:3b"
        if redocount > 30 or label==7:
            model_name = "ministral-3:3b"
        if redocount>150:
            break
        response = ollama.chat(
            model=model_name,#"hf.co/CrosswaveOmega/ministral8b-drmappy-mental-lora-unquantized-Q8_0-GGUF:Q8_0",
            
            messages=[
                {
                    "role": "system",
                    "content": ("When given the user's query, reword it to flow more naturally, keeping the exact defense names in quotes and the item numbers. "
                    "  You may summarize the text of the items together into around 3 sentences, but all items must be formatted as"+
                    " 'ITEM (numberhere)'.  "+
                    "Give your explanation the eyes of the eccentric 'Dr. Dennis Mappy,' a relaxed, fun-loving, and wise doctor of psychology whom specializes in explaining his logic in a fun way that is still simple and informative. Think Bob Ross if he were a psychiatrist.  You may ONLY use regular ascii characters, no emojis.  Keep it brief, around 3-5 sentences only.  ALL the mentioned defenses should be included, as well as ALL of the mentioned Item Numbers."+
                    "However, if it's level 0, it's best not to bring up any defense names or many item numbers.  If it's level 7, ALWAYS include the item numbers."
                    "If it's level 8, you MUST not include any defesnse names or items.  You must include the substring 'Needs More Evidence' case sensitive."
                    
                    )
                },
                
                {"role": "user", "content": f"The original question was \n{query}, \n the desired output is {expected_response}.  It's about a level {label}, the {label_to_name[label]} tier."}
    
            ],
            options={
                "temperature":0.5,
            })

        model_output = response["message"]["content"]
        pout = parse_paragraph(model_output)
        
        print(pout)
        post = post_process([pout])
        
        #print(post)
        continueiftrue=compare_dicts(compare_dict,pout)
        matched_label = next(iter(post["label_presence_all"]), None)
        if matched_label==label and continueiftrue:
            #print(f"Query:\n{query}\n")
            #print(expected_response)
            print(f"LABEL {label}: {i+lima}({b}/{N}) Model Response:\n{model_output[0:25]}\n")
            trainings.append((query,model_output))
            b+=1
            redocount=0
        else:
            print(i+lima,label,redocount,"REDO!")
            redocount+=1
    print("=" * 80)
    save_rows_to_csv_pandas(trainings)


defaultdict(<class 'list'>, {'Anticipation': ['ITEM_046'], 'Undoing': ['ITEM_083'], 'Suppression': ['ITEM_131'], 'Autistic fantasy': ['ITEM_002', 'ITEM_110'], 'Dissociation': ['ITEM_027', 'ITEM_073'], 'Passive aggression': ['ITEM_089'], 'Sublimation': ['ITEM_014', 'ITEM_036', 'ITEM_063'], 'Altruism': ['ITEM_079'], 'Omnipotence': ['ITEM_068'], 'Acting out': ['ITEM_005'], 'Repression': ['ITEM_136'], 'Humor': ['ITEM_051', 'ITEM_119']})
Anticipation ['ITEM_046']
Undoing ['ITEM_083']
Suppression ['ITEM_131']
Autistic fantasy ['ITEM_002', 'ITEM_110']
Dissociation ['ITEM_027', 'ITEM_073']
Passive aggression ['ITEM_089']
Sublimation ['ITEM_014', 'ITEM_036', 'ITEM_063']
Altruism ['ITEM_079']
Omnipotence ['ITEM_068']
Acting out ['ITEM_005']
Repression ['ITEM_136']
Humor ['ITEM_051', 'ITEM_119']
{'total_paragraphs': 1, 'defense_counts': {}, 'defense_rates': {}, 'index_counts': {}, 'index_rates': {}, 'item_counts': {}, 'item_rates': {}, 'rating_counts': {}, 'label_presence_all': {8: 1}, 'label_pre

KeyboardInterrupt: 